In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv('../data/processed/flights_cleaned.csv')
print(df.shape)
df.head()

C:\Users\hp\AppData\Local\Temp\ipykernel_20408\2889634024.py:4: DtypeWarning: Columns (0: ORIGIN_AIRPORT, 1: DESTINATION_AIRPORT) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/flights_cleaned.csv')


(5714008, 32)


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,IS_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0


In [3]:
mixed_origin = df[df['ORIGIN_AIRPORT'].str.isnumeric().fillna(False)]
print(f"Numeric origin codes: {mixed_origin.shape[0]}")
print(mixed_origin['ORIGIN_AIRPORT'].unique()[:10])

Numeric origin codes: 24126
['14747' '14771' '12889' '12892' '14869' '10299' '11292' '14107' '11630'
 '10732']


In [4]:
origin_mask = df['ORIGIN_AIRPORT'].astype(str).str.isnumeric().fillna(False)
dest_mask = df['DESTINATION_AIRPORT'].astype(str).str.isnumeric().fillna(False)

df = df[~(origin_mask | dest_mask)].copy()
print(df.shape)

(5231130, 32)


In [5]:
print("Origin numeric:", origin_mask.sum())
print("Dest numeric:", dest_mask.sum())
print("Either (dropped):", (origin_mask | dest_mask).sum())

Origin numeric: 482878
Dest numeric: 482878
Either (dropped): 482878


In [6]:
def hhmm_to_hour(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    return x // 100 # extract hour part from HHMM

df['SCHED_DEP_HOUR'] = df['SCHEDULED_DEPARTURE'].apply(hhmm_to_hour)
df['SCHED_ARR_HOUR'] = df['SCHEDULED_ARRIVAL'].apply(hhmm_to_hour)

print(df[['SCHEDULED_DEPARTURE', 'SCHED_DEP_HOUR', 'SCHEDULED_ARRIVAL', 'SCHED_ARR_HOUR']].head())

   SCHEDULED_DEPARTURE  SCHED_DEP_HOUR  SCHEDULED_ARRIVAL  SCHED_ARR_HOUR
0                    5               0                430               4
1                   10               0                750               7
2                   20               0                806               8
3                   20               0                805               8
4                   25               0                320               3


In [7]:
df['DEP_HOUR_SIN'] = np.sin(2 * np.pi * df['SCHED_DEP_HOUR'] / 24)
df['DEP_HOUR_COS'] = np.cos(2 * np.pi * df['SCHED_DEP_HOUR'] / 24)

print(df[['SCHED_DEP_HOUR', 'DEP_HOUR_SIN', 'DEP_HOUR_COS']].head(10))

   SCHED_DEP_HOUR  DEP_HOUR_SIN  DEP_HOUR_COS
0               0           0.0           1.0
1               0           0.0           1.0
2               0           0.0           1.0
3               0           0.0           1.0
4               0           0.0           1.0
5               0           0.0           1.0
6               0           0.0           1.0
7               0           0.0           1.0
8               0           0.0           1.0
9               0           0.0           1.0


In [8]:
import math
# Manual check for hour=8
print("Expected sin(8):", math.sin(2*math.pi*8/24))
print("Expected cos(8):", math.cos(2*math.pi*8/24))
print(df[df['SCHED_DEP_HOUR']==8][['SCHED_DEP_HOUR','DEP_HOUR_SIN','DEP_HOUR_COS']].head(2))

Expected sin(8): 0.8660254037844387
Expected cos(8): -0.4999999999999998
      SCHED_DEP_HOUR  DEP_HOUR_SIN  DEP_HOUR_COS
1442               8      0.866025          -0.5
1443               8      0.866025          -0.5


In [9]:
df['ARR_HOUR_SIN'] = np.sin(2 * np.pi * df['SCHED_ARR_HOUR'] / 24)
df['ARR_HOUR_COS'] = np.cos(2 * np.pi * df['SCHED_ARR_HOUR'] / 24)

In [10]:
df.to_csv('../data/processed/flights_day2.csv', index=False)
print("Saved!", df.shape)

Saved! (5231130, 38)
